# Hermes Agent — Explorer Notebook

This notebook lets you run, inspect, and test every part of Hermes interactively.

```
Hermes                        Redis (shared)              Icarus
------                        --------------              ------
Crawls external world  →→→   hermes:* namespace   ←←←   Pulls on demand
Stores structured data        hermes:item:*              Owns personal data
Runs on its own schedule      hermes:seen:*
Never touches personal data   hermes:supplier:*
```

**Sections:**
1. Setup
2. Supplier Config
3. RSS Crawler
4. Tavily Crawler
5. EDGAR Crawler
6. Signal Detector (Claude Haiku)
7. Redis — Read What's Stored
8. Full Pipeline (end-to-end)
9. Phase 1 Fixes
10. Miro Agent — Build Boards

---
## 1. Setup

Load environment variables and verify all keys are present.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

required_keys = [
    "ANTHROPIC_API_KEY",
    "TAVILY_API_KEY",
    "UPSTASH_REDIS_REST_URL",
    "UPSTASH_REDIS_REST_TOKEN",
    "MIRO_API_TOKEN",
]

for key in required_keys:
    val = os.environ.get(key, "")
    status = "✅" if val else "❌ MISSING"
    print(f"{status}  {key}: {val[:20]}..." if val else f"{status}  {key}")

---
## 2. Supplier Config

~250 companies across 17+ categories. Each has a name, tier, category, and optional RSS feed.

In [ ]:
from config.suppliers import ALL_SUPPLIERS

print(f"Total suppliers: {len(ALL_SUPPLIERS)}")

# Count by category
from collections import Counter
cats = Counter(s.get("category", "Unknown") for s in ALL_SUPPLIERS)
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {count:3d}  {cat}")

In [ ]:
# Count by tier
tiers = Counter(s.get("tier", 0) for s in ALL_SUPPLIERS)
for tier, count in sorted(tiers.items()):
    print(f"  Tier {tier}: {count} suppliers")

# How many have RSS feeds?
with_rss = sum(1 for s in ALL_SUPPLIERS if s.get("rss"))
print(f"\nWith RSS feeds: {with_rss}/{len(ALL_SUPPLIERS)}")

In [ ]:
# Inspect a specific category — change this to explore
CATEGORY = "AI Foundation Labs"

filtered = [s for s in ALL_SUPPLIERS if s.get("category") == CATEGORY]
for s in filtered:
    rss = "📡 RSS" if s.get("rss") else "     "
    print(f"  [{s.get('tier', '?')}] {rss}  {s['name']}")

---
## 3. RSS Crawler

Parses RSS/Atom feeds for all suppliers that have one. Runs every 6 hours in production.

In this notebook we run it against a **dry Redis store** so nothing is permanently marked as seen.

In [ ]:
# Dry-run store — marks nothing as seen, stores nothing
class DryRunStore:
    def is_seen(self, item_id): return False
    def mark_seen(self, item_id): pass
    def store_items(self, items): pass

dry_store = DryRunStore()

In [ ]:
# Test a single RSS feed without touching real Redis
import feedparser
import hashlib
from datetime import datetime, timezone

def parse_feed_preview(feed_url: str, supplier_name: str, limit: int = 5):
    feed = feedparser.parse(feed_url)
    print(f"Feed: {feed.feed.get('title', 'unknown')} ({len(feed.entries)} entries)")
    for entry in feed.entries[:limit]:
        print(f"  • {entry.get('title', 'no title')[:80]}")
        print(f"    {entry.get('link', '')[:70]}")

# Try one supplier — find one with an RSS feed
rss_suppliers = [s for s in ALL_SUPPLIERS if s.get("rss")]
example = rss_suppliers[0]  # change index to try different ones
print(f"Testing: {example['name']}")
print(f"URL: {example['rss']}\n")
parse_feed_preview(example["rss"], example["name"])

In [ ]:
# Run the full RSS crawler (dry run — no Redis writes)
from crawlers.rss_crawler import crawl_rss

rss_items = crawl_rss(dry_store)
print(f"\nTotal new RSS items found: {len(rss_items)}")

# Show a sample
for item in rss_items[:5]:
    print(f"\n  [{item['supplier']}]")
    print(f"  Title: {item['title'][:80]}")
    print(f"  URL:   {item['url'][:70]}")

---
## 4. Tavily Crawler

Uses Tavily's news search API to find recent articles about each tracked company.
Tier 1+2 run daily; all tiers run weekly.

In [ ]:
from crawlers.tavily_crawler import crawl_tavily

# Run Tier 1 only to keep API costs low during testing
# Change tier=1 to tier=2 or tier=3 for broader coverage
tavily_items = crawl_tavily(dry_store, tier=1)
print(f"\nTotal new Tavily items found: {len(tavily_items)}")

for item in tavily_items[:5]:
    print(f"\n  [{item['supplier']}]")
    print(f"  Title:   {item['title'][:80]}")
    print(f"  Summary: {item.get('summary', '')[:120]}...")

---
## 5. EDGAR Crawler

Fetches SEC filings (10-K, 10-Q, 8-K) for US-listed Tier 1+2 companies.
No API key needed — EDGAR is public.

In [ ]:
from crawlers.edgar_crawler import crawl_edgar

edgar_items = crawl_edgar(dry_store)
print(f"\nTotal new EDGAR items found: {len(edgar_items)}")

for item in edgar_items[:5]:
    print(f"\n  [{item['supplier']}]")
    print(f"  Title: {item['title'][:80]}")
    print(f"  URL:   {item['url'][:70]}")

---
## 6. Signal Detector (Claude Haiku)

Each news item is sent to Claude Haiku which classifies it into a signal type and rates urgency.

**Signal types:** FUNDING 💰 · ACQUISITION 🤝 · PRODUCT_RELEASE 🆕 · PRICING_CHANGE 💲 · SUPPLY_CHAIN ⚠️ · EARNINGS 📊 · PARTNERSHIP 🔗 · REGULATORY ⚖️ · LAYOFFS_HIRING 👥 · RESEARCH_PAPER 🔬 · OTHER 📰

In [ ]:
# Test signal detection on a synthetic item (no crawling needed)
from processors.signal_detector import detect_signals

test_items = [
    {
        "id": "test_001",
        "supplier": "NVIDIA",
        "title": "NVIDIA announces $500M Series D for new AI chip startup acquisition",
        "summary": "NVIDIA has completed its acquisition of a semiconductor startup for $500 million, gaining access to next-generation chip designs.",
        "url": "https://example.com/nvidia-acquisition",
        "published": "2026-05-04T10:00:00Z",
        "source": "test",
    },
    {
        "id": "test_002",
        "supplier": "OpenAI",
        "title": "OpenAI releases GPT-5 with 10x performance improvement",
        "summary": "OpenAI today released GPT-5, claiming major improvements in reasoning and coding tasks over previous models.",
        "url": "https://example.com/gpt5",
        "published": "2026-05-04T09:00:00Z",
        "source": "test",
    },
    {
        "id": "test_003",
        "supplier": "TSMC",
        "title": "TSMC opens new Arizona fab ahead of schedule",
        "summary": "TSMC's Arizona fabrication plant began production two quarters ahead of schedule, adding capacity for US chip manufacturing.",
        "url": "https://example.com/tsmc-az",
        "published": "2026-05-04T08:00:00Z",
        "source": "test",
    },
]

enriched = detect_signals(test_items)

for item in enriched:
    sig = "🔔 SIGNIFICANT" if item["is_significant"] else "   routine"
    print(f"\n{item['emoji']} [{item['signal_type']}] {sig} ({item['urgency']})")
    print(f"   {item['supplier']}: {item['title'][:70]}")
    if item.get("significance_reason"):
        print(f"   Reason: {item['significance_reason']}")

---
## 7. Redis — Read What's Stored

Connect to the live Upstash Redis and inspect what Hermes has already collected.

In [ ]:
from storage.redis_store import RedisStore

store = RedisStore()
print("✅ Connected to Redis")

In [ ]:
# Count all hermes:* keys
all_keys = store.r.keys("hermes:*")
items  = [k for k in all_keys if k.startswith("hermes:item:")]
seen   = [k for k in all_keys if k.startswith("hermes:seen:")]
suppliers = [k for k in all_keys if k.startswith("hermes:supplier:")]

print(f"hermes:item:*     {len(items):4d}  (full news items, 7-day TTL)")
print(f"hermes:seen:*     {len(seen):4d}  (dedup flags, 30-day TTL)")
print(f"hermes:supplier:* {len(suppliers):4d}  (per-supplier item ID lists)")

In [ ]:
# Look up a specific supplier — change this to any tracked company
SUPPLIER = "nvidia"  # lowercase, spaces → underscores

items = store.get_supplier_items(SUPPLIER, limit=10)
print(f"Last {len(items)} items for '{SUPPLIER}':\n")

for item in items:
    emoji = item.get("emoji", "📰")
    sig = "🔔" if item.get("is_significant") else "  "
    print(f"{sig} {emoji} [{item.get('signal_type', 'OTHER')}] {item['published'][:10]}")
    print(f"     {item['title'][:80]}")
    print(f"     {item.get('url', '')[:70]}")
    print()

In [ ]:
# Get significant items across all suppliers
significant = store.get_significant_items(limit=20)
print(f"Significant items in Redis: {len(significant)}\n")

for item in significant:
    print(f"{item['emoji']} [{item['urgency']}] {item['supplier']} — {item['title'][:60]}")
    print(f"   {item.get('significance_reason', '')[:100]}")
    print()

In [ ]:
# Signal type breakdown of everything in Redis
import json
from collections import Counter

all_item_keys = store.r.keys("hermes:item:*")
signal_counts = Counter()
urgency_counts = Counter()

for key in all_item_keys:
    raw = store.r.get(key)
    if raw:
        item = json.loads(raw)
        signal_counts[item.get("signal_type", "OTHER")] += 1
        if item.get("is_significant"):
            urgency_counts[item.get("urgency", "LOW")] += 1

print("Signal type breakdown:")
for sig, count in signal_counts.most_common():
    print(f"  {count:4d}  {sig}")

print("\nSignificant items by urgency:")
for urgency, count in urgency_counts.most_common():
    print(f"  {count:4d}  {urgency}")

---
## 8. Full Pipeline (End-to-End)

Run a full cycle: crawl → detect signals → store in Redis.

⚠️ This writes to Redis and costs Tavily + Anthropic API calls. Use for real runs.

In [ ]:
# Full RSS pipeline (real Redis, real API calls)
# Comment out the raise to actually run it
raise Exception("Uncomment to run a real RSS cycle")

from crawlers.rss_crawler import crawl_rss
from processors.signal_detector import detect_signals

store = RedisStore()
items = crawl_rss(store)
if items:
    enriched = detect_signals(items)
    store.store_items(enriched)
    significant = [i for i in enriched if i["is_significant"]]
    print(f"Stored {len(enriched)} items, {len(significant)} significant")
else:
    print("No new RSS items")

In [ ]:
# Full Tavily pipeline — Tier 1 only
# Comment out the raise to actually run it
raise Exception("Uncomment to run a real Tavily cycle (costs API credits)")

from crawlers.tavily_crawler import crawl_tavily

store = RedisStore()
items = crawl_tavily(store, tier=1)
if items:
    enriched = detect_signals(items)
    store.store_items(enriched)
    significant = [i for i in enriched if i["is_significant"]]
    print(f"Stored {len(enriched)} items, {len(significant)} significant")
else:
    print("No new Tavily items")

---
## 9. Phase 1 Fixes

Two things need fixing before Hermes deploys:

### Fix A — Remove Telegram notifier
Hermes should never push. The `notifier/` module violates the architecture. `main.py` currently imports and calls it on every cycle.

### Fix B — Remove Telegram env vars from `.env.example`
Hermes has no business knowing the Telegram bot token.

In [ ]:
# Verify the problem: main.py still imports the notifier
with open("main.py") as f:
    content = f.read()

lines_with_notifier = [(i+1, line.rstrip()) for i, line in enumerate(content.splitlines()) if "notif" in line.lower()]
print("Lines in main.py referencing notifier:")
for lineno, line in lines_with_notifier:
    print(f"  {lineno:3d}: {line}")

In [ ]:
# Preview the clean version of main.py (no notifier)
clean_main = '''\
import os
from apscheduler.schedulers.blocking import BlockingScheduler
from apscheduler.triggers.cron import CronTrigger

from config.suppliers import ALL_SUPPLIERS
from crawlers.rss_crawler import crawl_rss
from crawlers.tavily_crawler import crawl_tavily
from crawlers.edgar_crawler import crawl_edgar
from processors.signal_detector import detect_signals
from storage.redis_store import RedisStore


store = RedisStore()


def run_rss_cycle():
    print("[Main] Running RSS cycle...")
    items = crawl_rss(store)
    if items:
        enriched = detect_signals(items)
        store.store_items(enriched)


def run_tavily_cycle():
    print("[Main] Running Tavily cycle (Tier 1+2)...")
    items = crawl_tavily(store, tier=2)
    if items:
        enriched = detect_signals(items)
        store.store_items(enriched)


def run_edgar_cycle():
    print("[Main] Running EDGAR cycle...")
    items = crawl_edgar(store)
    if items:
        enriched = detect_signals(items)
        store.store_items(enriched)


def run_full_cycle():
    print("[Main] Running full cycle (all tiers)...")
    items = crawl_tavily(store, tier=3)
    if items:
        enriched = detect_signals(items)
        store.store_items(enriched)


if __name__ == "__main__":
    print(f"[Hermes] Starting — monitoring {len(ALL_SUPPLIERS)} suppliers")

    run_rss_cycle()
    run_tavily_cycle()
    run_edgar_cycle()

    scheduler = BlockingScheduler(timezone="Europe/Berlin")
    scheduler.add_job(run_rss_cycle,    CronTrigger(hour="0,6,12,18", minute=0))
    scheduler.add_job(run_edgar_cycle,  CronTrigger(hour="7", minute=30))
    scheduler.add_job(run_tavily_cycle, CronTrigger(hour="8", minute=0))
    scheduler.add_job(run_full_cycle,   CronTrigger(day_of_week="mon", hour="9", minute=0))

    print("[Hermes] Scheduler running")
    scheduler.start()
'''

print(clean_main)

In [ ]:
# Apply Fix A — write the clean main.py
# Remove the raise to apply
raise Exception("Uncomment to apply Fix A: remove Telegram notifier from main.py")

with open("main.py", "w") as f:
    f.write(clean_main)
print("✅ main.py updated — notifier removed")

In [ ]:
# Apply Fix B — clean up .env.example
# Remove the raise to apply
raise Exception("Uncomment to apply Fix B: remove Telegram vars from .env.example")

clean_env = """ANTHROPIC_API_KEY=your_anthropic_api_key
TAVILY_API_KEY=your_tavily_api_key
UPSTASH_REDIS_REST_URL=your_upstash_redis_url
UPSTASH_REDIS_REST_TOKEN=your_upstash_redis_token
"""

with open(".env.example", "w") as f:
    f.write(clean_env)
print("✅ .env.example updated — Telegram vars removed")

In [ ]:
# Verify both fixes
import subprocess

# Check main.py has no notifier references
with open("main.py") as f:
    main_content = f.read()

if "notif" in main_content.lower():
    print("❌ main.py still references notifier")
else:
    print("✅ main.py: no notifier references")

# Check .env.example has no Telegram vars
with open(".env.example") as f:
    env_content = f.read()

if "TELEGRAM" in env_content:
    print("❌ .env.example still has TELEGRAM vars")
else:
    print("✅ .env.example: no TELEGRAM vars")

# Import check — does clean main.py import cleanly?
result = subprocess.run(["python", "-c", "import main"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ main.py imports without errors")
else:
    print(f"❌ Import error: {result.stderr[:200]}")

---
## Roadmap Tracker

| Phase | Task | Status |
|-------|------|--------|
| 1 | Remove Telegram notifier | ✅ Done |
| 1 | Create GitHub repo `hermes-agent` | ✅ Done |
| 1 | Deploy to Railway | ✅ Done |
| 1 | Set env vars on Railway | ✅ Done |
| 1 | Verify crawlers run on schedule | ⬜ Check Railway logs |
| 1 | Confirm Redis keys write correctly | ⬜ Section 7 above |
| 2 | Icarus: query `hermes:supplier:{slug}` | ⬜ |
| 2 | Icarus: daily signal briefing command | ⬜ |
| 3 | Company profile schema design | ⬜ |
| 3 | Populate `hermes:profile:{slug}` | ⬜ |
| 4 | Miro Agent — signal board | ✅ Section 10 below |
| 4 | Miro Agent — landscape board | ✅ Section 10 below |
| 4 | Miro Agent — Icarus Telegram trigger | ⬜ |

---
## 10. Miro Agent — Build Boards

Two board types available:

- **Signal Board** — significant Hermes items as sticky notes, grouped by signal type, colored by urgency (🔴 HIGH · 🟡 MEDIUM · 🟢 LOW)
- **Landscape Board** — all tracked suppliers as cards, grouped by category, colored by tier (🔴 Tier 1 · 🟠 Tier 2 · ⚫ Tier 3)

Boards are created in your Miro account and a link is returned.

In [ ]:
# Connect to Redis and verify Miro token
from storage.redis_store import RedisStore
from miro.client import MiroClient

store = RedisStore()
client = MiroClient()

# Quick connection test
import httpx
r = httpx.get("https://api.miro.com/v2/boards", headers={"Authorization": f"Bearer {client.token}"})
print(f"Redis: {'✅' if store.r.keys('hermes:*') is not None else '❌'}")
print(f"Miro:  {'✅' if r.status_code == 200 else '❌'} ({r.status_code})")

# How many significant items are available for the signal board?
sig_items = store.get_significant_items(limit=100)
print(f"\nSignificant items in Redis: {len(sig_items)}")
print("(Run crawlers in Section 8 first if this is 0)")

In [ ]:
# Build a Signal Board — significant Hermes items grouped by signal type
# Creates a real Miro board in your account
from miro.boards import build_signal_board

url = build_signal_board(store)
print(f"\nBoard URL: {url}")

In [ ]:
# Build a Landscape Board — all suppliers grouped by category
# Pass category_filter to limit to one category, or leave None for all
from miro.boards import build_landscape_board

CATEGORY = None  # e.g. "AI Foundation Labs" or "Semiconductors & Chips" — None = all

url = build_landscape_board(store, category_filter=CATEGORY)
print(f"\nBoard URL: {url}")

---
## 11. SpendLens Integration — HermesClient

The `HermesClient` is a standalone module SpendLens can drop in to pull Hermes intelligence for any vendor in its spend data.

**Procurement-relevant signals:** SUPPLY_CHAIN ⚠️ · PRICING_CHANGE 💲 · EARNINGS 📊 · REGULATORY ⚖️ · ACQUISITION 🤝 · LAYOFFS_HIRING 👥

**Key features:**
- Fuzzy vendor name matching (e.g. "Taiwan Semiconductor" → finds "tsmc")
- `get_signals(vendor)` — recent signals for one vendor
- `get_risk_flags(vendor)` — significant HIGH/MEDIUM signals only
- `enrich_vendor_list([...])` — bulk risk summary for a full vendor list
- `get_procurement_briefing()` — top signals across all tracked suppliers

In [ ]:
from integrations.hermes_client import HermesClient

hermes = HermesClient()
print("✅ HermesClient connected")

# Show all supplier slugs currently in Redis
slugs = hermes._known_slugs()
print(f"Tracked suppliers with data in Redis: {len(slugs)}")
for s in slugs[:20]:
    print(f"  {s}")

In [ ]:
# Get signals for a single vendor — try fuzzy matching too
VENDOR = "TSMC"  # change to any vendor name

signals = hermes.get_signals(VENDOR, limit=10)
print(f"Signals for '{VENDOR}': {len(signals)} found\n")
for item in signals:
    print(hermes.format_signal_for_display(item))
    print()

In [ ]:
# Simulate a SpendLens vendor list — bulk risk enrichment
# Replace with real vendor names from your SpendLens spend data
VENDOR_LIST = [
    "TSMC",
    "NVIDIA",
    "Intel",
    "Arrow Electronics",
    "Samsung Electronics",
    "Cisco Systems",
    "Some Random Vendor XYZ",  # not tracked — shows UNKNOWN handling
]

risk_map = hermes.enrich_vendor_list(VENDOR_LIST)

print(f"{'Vendor':<30} {'Tracked':<10} {'Risk':<10} {'Signals':<10} Top Signal")
print("-" * 90)
for vendor, data in risk_map.items():
    tracked = "✅" if data["tracked"] else "❌ not tracked"
    risk = data["risk_level"]
    count = data["signal_count"]
    top = data["top_signal"]
    top_str = f"{top.get('emoji','')} {top.get('signal_type','')} — {top.get('title','')[:35]}..." if top else "—"
    print(f"{vendor:<30} {tracked:<10} {risk:<10} {count:<10} {top_str}")

In [ ]:
# Procurement briefing — top signals across all tracked suppliers
# This is what SpendLens market intelligence tab would show
briefing = hermes.get_procurement_briefing(limit=20)
print(f"Top procurement signals in Hermes ({len(briefing)} items):\n")
for item in briefing:
    print(hermes.format_signal_for_display(item))
    print()